# Coinbase AI Trader — Cloud CNN Retrain (Colab)

Runs `tools/train_cloud.py` on a Colab GPU (T4/A100) to retrain the production CNN faster than the local RTX 2060.

**Workflow:**
1. Verify GPU.
2. Bring in the repo (git clone) and the dataset cache (Drive or upload).
3. Install minimal deps (`tools/requirements-train.txt`).
4. Run `tools/train_cloud.py`.
5. Download `cnn_model.pt`, `cnn_best_loss.txt`, `train_cloud_progress.json` and drop them into the local `backend/` directory; the trader hot-reloads on the next restart (or via the planned `POST /api/cnn/model/reload` endpoint, #69).

**Note on Ch 20 (funding rate):** Binance `/fapi/v1/fundingRate` is geoblocked from US IPs but works from Colab. v1 of this notebook trains on whatever cache you upload; v2 will refetch funding history in-Colab and rebuild Ch 20 before training.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Get the repo

Replace the URL below with your own remote. If the repo is private, use a deploy-key/PAT pattern (`https://<token>@github.com/<user>/<repo>.git`).

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/gl4500/polymarket_app.git"  # TODO: set
REPO_DIR = "/content/polymarket_app"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 3. Get the dataset cache

Pick one of the two cells below — Drive mount or direct upload.

In [ ]:
# Option A — Google Drive mount
# Place backend/cnn_dataset_cache.pt at MyDrive/coinbase_trader/cnn_dataset_cache.pt
from google.colab import drive
drive.mount('/content/drive')
import shutil
src = '/content/drive/MyDrive/coinbase_trader/cnn_dataset_cache.pt'
dst = 'backend/cnn_dataset_cache.pt'
os.makedirs('backend', exist_ok=True)
shutil.copy(src, dst)
print('cache size MB:', round(os.path.getsize(dst) / 1e6, 1))

In [ ]:
# Option B — direct upload (skip if you used Drive)
# from google.colab import files
# uploaded = files.upload()  # pick cnn_dataset_cache.pt
# import shutil; os.makedirs('backend', exist_ok=True)
# for fname in uploaded: shutil.move(fname, f'backend/{fname}')

## 4. Install deps

In [ ]:
!pip install -q -r tools/requirements-train.txt
import torch
print('torch=', torch.__version__, 'cuda=', torch.cuda.is_available())

## 5. Train

16 epochs ≈ 1–3 min on T4 vs ~9 min on RTX 2060. Tune `--epochs` if you want longer runs.

In [ ]:
!python tools/train_cloud.py \
    --cache backend/cnn_dataset_cache.pt \
    --out-model backend/cnn_model.pt \
    --out-best-loss backend/cnn_best_loss.txt \
    --out-progress backend/train_cloud_progress.json \
    --epochs 16 --seed 42 --val-frac 0.2

In [ ]:
import json
print(json.dumps(json.load(open('backend/train_cloud_progress.json')), indent=2))

## 6. Download artifacts

Drop the three files below into `C:\Users\gl450\polymarket_app\backend\` on your local machine.

In [ ]:
from google.colab import files
for f in ['backend/cnn_model.pt', 'backend/cnn_best_loss.txt', 'backend/train_cloud_progress.json']:
    files.download(f)